# 04 — Final held-out evaluation

This notebook applies the validation-frozen analysis plan to the held-out test cohort. The safe
default is review-only: `RUN_FINAL_EVALUATION=False` loads only small saved JSON/CSV reports. A new
inference requires explicit opt-in, the canonical validation thresholds, and a separate overwrite
confirmation if predictions already exist.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))

# Execution contract: offline and non-mutating unless deliberately opted into.
# See notebooks/utility/review_mode.py for the flags and the environment overrides.
import review_mode
ALLOW_NETWORK_ACCESS = False
INSTALL_DEPENDENCIES = False
ALLOW_PROCESSED_DOWNLOAD = False
review_mode.activate(
    allow_network=ALLOW_NETWORK_ACCESS,
    allow_dependency_install=INSTALL_DEPENDENCIES,
    allow_processed_download=ALLOW_PROCESSED_DOWNLOAD,
)

from notebooks.utility.classifier_analysis import (
    ensemble_metric_table,
    plot_ensemble_curves,
    plot_ensemble_overview,
)
from notebooks.utility.classifier_protocol import ARCHITECTURES, CONDITIONS, SEEDS
from notebooks.utility.final_evaluation import EXPECTED_PATIENT_COUNT, require_final_evaluation_opt_in
from notebooks.utility.regenerate_classifier_metrics import regenerate

RUN_FINAL_EVALUATION = False
OVERWRITE_TEST_PREDICTIONS = False
{
    'run_final_evaluation': RUN_FINAL_EVALUATION,
    'overwrite_test_predictions': OVERWRITE_TEST_PREDICTIONS,
    'expected_patient_count': EXPECTED_PATIENT_COUNT,
    'mode': 'review-only' if not RUN_FINAL_EVALUATION else 'inference enabled',
}

## 1. Apply guards before any test or model access

Review mode does not open test metadata, checkpoints, adapters, GPU libraries or models. The explicit
final-evaluation branch first requires all canonical validation thresholds and checks existing predictions.

In [ ]:
test_data = None
if RUN_FINAL_EVALUATION:
    require_final_evaluation_opt_in(
        ROOT,
        run_final_evaluation=RUN_FINAL_EVALUATION,
        overwrite_test_predictions=OVERWRITE_TEST_PREDICTIONS,
    )
    from notebooks.utility.classifier_dataset_builder import test_rows
    test_data = test_rows(ROOT)
    if len({row['patient_id'] for row in test_data}) != EXPECTED_PATIENT_COUNT:
        raise ValueError('test patient count differs from the expected final cohort')
    test_inventory = {
        'test_images': len(test_data),
        'test_patients': len({row['patient_id'] for row in test_data}),
        'positive_images': sum(1 for row in test_data if row['label'] == 1),
    }
else:
    test_inventory = {'status': 'review-only; test metadata was not opened'}
test_inventory

## 2. Optional inference over the frozen 24-job matrix

The branch is disabled by default. Each seed uses the decision and target-specificity thresholds
stored in its `validation_metrics.json`; existing test predictions are never overwritten unless
`OVERWRITE_TEST_PREDICTIONS=True` is set separately.

In [ ]:
if RUN_FINAL_EVALUATION:
    import gc
    import torch
    from notebooks.utility.classifier_experiment import (
        configure_environment,
        experiment_configuration,
        load_existing_outputs,
        run_test,
    )

    base_configuration = experiment_configuration(ROOT, ARCHITECTURES[0], CONDITIONS[0], SEEDS[0])
    base_configuration['root'] = str(ROOT)
    configure_environment(base_configuration)
    for architecture in ARCHITECTURES:
        for condition in CONDITIONS:
            for seed in SEEDS:
                configuration = experiment_configuration(ROOT, architecture, condition, seed)
                configuration['root'] = str(ROOT)
                checkpoint = load_existing_outputs(ROOT, configuration)['checkpoint']
                if checkpoint is None:
                    raise RuntimeError(f'No trained checkpoint for {architecture}/{condition}/seed_{seed}.')
                result = run_test(
                    ROOT, configuration, checkpoint, test_data,
                    overwrite_test_predictions=OVERWRITE_TEST_PREDICTIONS,
                )
                print(architecture, condition, seed, 'test PR-AUC', round(result['metrics']['pr_auc'], 4))
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
else:
    print('Final inference disabled: review-only mode did not load a model, GPU, checkpoint or test row.')

## 3. Reconstruct the eight frozen three-seed test ensembles

The same architecture–condition seed membership used on validation is applied to the aligned test
predictions. Probabilities are averaged across the three seeds at image level and then within patient,
with strict key, label, manifest, duplicate, and finiteness checks. Ensemble composition cannot be
changed in response to test performance.

In [ ]:
if RUN_FINAL_EVALUATION:
    regeneration_summary = regenerate(ROOT)
    print(regeneration_summary)
test_ensembles = [
    json.loads((ROOT / 'results/4_final_evaluation/test_ensembles' / architecture / condition / 'ensemble_metrics.json').read_text())
    for architecture in ARCHITECTURES
    for condition in CONDITIONS
]
len(test_ensembles)

## 4. Estimate patient-level held-out performance

The final table reports the registered patient-level discrimination, Brier, and calibration metrics
with patient-resampled confidence intervals. These are the primary held-out estimates for the eight
frozen ensembles. Their uncertainty pertains to this cohort and should not be extrapolated to other
institutions or acquisition protocols without external validation.

In [ ]:
test_comparison = json.loads((ROOT / 'results/4_final_evaluation/results.json').read_text())
ensemble_metric_table(test_ensembles)

## 5. Apply the protocol-declared Holm family of eight comparisons

The exact comparison family defined on validation is evaluated on paired test-patient predictions:
three augmentation-versus-real-only contrasts and one fine-tuned-versus-from-scratch contrast within
each architecture. Holm adjustment is applied jointly to the eight p-values. No hypothesis is added,
removed, or reordered after observing held-out outcomes.

In [ ]:
test_comparison['comparisons'], test_comparison['holm_correction']

## 6. Render final descriptive figures

Overview, PR/ROC, and calibration figures summarize the frozen test ensembles and their uncertainty.
The plotting step reads verified predictions and does not alter metrics, thresholds, or statistical
decisions. Figures are descriptive companions to the tabulated, multiplicity-controlled analysis and
should be interpreted together with cohort size and confidence intervals.

In [ ]:
overview_figure = plot_ensemble_overview(test_ensembles, split='test')
curve_figure = plot_ensemble_curves(ROOT, test_ensembles, split='test')
(overview_figure, curve_figure)